# Fine-tuning LoRA — Phi-3.5-mini sur dataset médical
TechCorp Industries — Mission Expérimentale IA

## 1. Installation des dépendances

In [ ]:
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

## 2. Vérification du GPU

In [ ]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun GPU détecté")

## 3. Chargement du modèle Phi-3.5-mini en 4-bit

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 4. Configuration LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Upload et préparation du dataset médical
Uploadez `medical_dataset_final.json` depuis votre Google Drive ou directement via le bouton d'upload de fichier Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Sélectionnez medical_dataset_final.json

In [ ]:
import json
import random

with open("medical_dataset_final.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Echantillonnage pour rester compatible avec un GPU Colab gratuit/Pro
SAMPLE_SIZE = 5000
random.seed(42)
sample = random.sample(data, min(SAMPLE_SIZE, len(data)))
print(f"Exemples utilises pour l'entrainement: {len(sample)}")

In [ ]:
from datasets import Dataset

def format_prompt(example):
    text = f"""### Question du patient:
{example['user']}

### Reponse du medecin:
{example['assistant']}<|endoftext|>"""
    return {"text": text}

formatted = [format_prompt(ex) for ex in sample]
dataset = Dataset.from_list(formatted)
print(dataset[0]["text"][:300])

## 6. Configuration et lancement de l'entraînement

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "no",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = True,
    ),
)

trainer_stats = trainer.train()

## 7. Sauvegarde des métriques d'entraînement (loss, epochs)

In [ ]:
import json

log_history = trainer.state.log_history
with open("training_metrics.json", "w") as f:
    json.dump(log_history, f, indent=2)

for entry in log_history:
    if "loss" in entry:
        print(f"Epoch: {entry.get('epoch', 'N/A'):.2f} | Loss: {entry['loss']:.4f}")

## 8. Test rapide du modèle fine-tuné

In [ ]:
FastLanguageModel.for_inference(model)

test_question = "I have a persistent headache and feel dizzy, what could it be?"
prompt = f"""### Question du patient:
{test_question}

### Reponse du medecin:
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

## 9. Sauvegarde du modèle LoRA fine-tuné

In [ ]:
model.save_pretrained("phi3.5_medical_lora")
tokenizer.save_pretrained("phi3.5_medical_lora")

from google.colab import files
!zip -r phi3.5_medical_lora.zip phi3.5_medical_lora
files.download("phi3.5_medical_lora.zip")
files.download("training_metrics.json")